# NLP + Linear SVM for Complaint Classification
### Broad classification + specific category | Bisaya/Cebuano and English

This notebook trains **two separate text classifiers** using `COMPLAINTS_HISTORY_AND_CATEGORIZATION.xlsx`:

1. **Broad classification** — e.g., CIVIL, CRIMINAL, Administrative.
2. **Specific category** — predicts the detailed complaint category.

**Pipeline:** complaint text → word + character TF-IDF features → Linear SVM (`LinearSVC`) → predictions and evaluation.

The dataset has 600 records, including Bisaya and English complaints. The notebook keeps the original text and does not translate it, so it can learn from the language and spelling patterns present in the dataset.

> **Responsible use:** Treat predictions as decision support, not a final legal determination. Review errors, especially for rare categories. Do not upload real complainant data to a shared notebook unless you are authorized to process it.

## 1. Install and import libraries

In [ ]:
!pip -q install pandas openpyxl scikit-learn joblib matplotlib seaborn

In [ ]:
import os
import json
import shutil
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix
)

RANDOM_STATE = 42
TEST_SIZE = 0.20

print("Libraries ready.")

## 2. Upload the Excel dataset

Run the cell and select `COMPLAINTS_HISTORY_AND_CATEGORIZATION.xlsx` from your computer.

In [ ]:
uploaded = files.upload()
if not uploaded:
    raise ValueError("No file was uploaded.")

excel_name = next(iter(uploaded))
df = pd.read_excel(excel_name)

print("File:", excel_name)
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
display(df.head())

## 3. Validate and prepare the data

Required fields are `complaint_text`, `classification`, and `category`. `record_id`, `language`, and `verified` are retained for reference but are **not** model input features.

The model uses the original complaint text. This avoids accidentally learning from IDs or from metadata that may not be available when a new complaint arrives.

In [ ]:
required_columns = ["complaint_text", "classification", "category"]
missing = [col for col in required_columns if col not in df.columns]
if missing:
    raise ValueError(
        f"Missing required columns: {missing}. "
        f"Available columns: {df.columns.tolist()}"
    )

data = df.copy()

# Normalize text fields while preserving Bisaya/Cebuano and English wording.
for col in required_columns:
    data[col] = data[col].fillna("").astype(str).str.strip()

before = len(data)
data = data[
    data["complaint_text"].ne("") &
    data["classification"].ne("") &
    data["category"].ne("")
].copy()
data = data.reset_index(drop=True)

print(f"Rows before cleaning: {before}")
print(f"Rows after removing blank text/labels: {len(data)}")
print("\nBroad classification counts:")
display(data["classification"].value_counts().rename_axis("classification").to_frame("records"))

print("\nSpecific category counts:")
display(data["category"].value_counts().rename_axis("category").to_frame("records"))

if data["classification"].nunique() < 2:
    raise ValueError("The broad classification target must contain at least two classes.")
if data["category"].nunique() < 2:
    raise ValueError("The specific category target must contain at least two classes.")

print("\nLanguage field counts (if present):")
if "language" in data.columns:
    display(data["language"].fillna("Unknown").value_counts().to_frame("records"))

## 4. Create the NLP + Linear SVM pipeline

- **Word TF-IDF (1–2 grams):** learns individual words and short phrases.
- **Character TF-IDF (3–5 grams):** can help with spelling variation, informal phrasing, and mixed-language text.
- **LinearSVC:** learns a decision boundary between the labels.
- `class_weight="balanced"` helps give less frequent labels more influence during training.

The vectorizers are fitted only on the training split, preventing test text from influencing vocabulary fitting.

In [ ]:
def build_model():
    text_features = FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),
            min_df=1,
            max_features=50000,
            sublinear_tf=True
        )),
        ("char_tfidf", TfidfVectorizer(
            analyzer="char",
            ngram_range=(3, 5),
            min_df=1,
            max_features=50000,
            sublinear_tf=True
        ))
    ])

    return Pipeline([
        ("features", text_features),
        ("svm", LinearSVC(
            C=1.0,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            max_iter=10000
        ))
    ])

print("Pipeline function ready.")

## 5. Train both models and evaluate on a held-out test set

Each model uses an 80/20 stratified split when every class has at least two records. If a target contains a class with only one example, stratification is not possible and the notebook stops with a clear message rather than silently producing an invalid evaluation.

The same split indices are used for both targets, making their evaluation comparable.

In [ ]:
X = data["complaint_text"].astype(str)

targets = {
    "broad_classification": data["classification"].astype(str),
    "specific_category": data["category"].astype(str)
}

# Check rare classes before a stratified split.
for target_name, y in targets.items():
    rare_counts = y.value_counts()
    singletons = rare_counts[rare_counts < 2]
    if len(singletons):
        raise ValueError(
            f"{target_name} has labels with fewer than 2 records: "
            f"{singletons.to_dict()}. Add more labeled examples or merge/review "
            "those labels before using a stratified train/test evaluation."
        )

indices = np.arange(len(data))
train_idx, test_idx = train_test_split(
    indices,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=targets["specific_category"]
)

# Check that the chosen split also includes every broad-class label in train and test.
for target_name, y in targets.items():
    missing_train = set(y.iloc[test_idx].unique()) - set(y.iloc[train_idx].unique())
    if missing_train:
        raise ValueError(
            f"Split issue for {target_name}; labels absent from training: {missing_train}"
        )

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

models = {}
evaluation = {}
summary_rows = []

for target_name, y in targets.items():
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]

    model = build_model()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    models[target_name] = model
    evaluation[target_name] = {
        "y_test": y_test,
        "y_pred": pd.Series(y_pred, index=y_test.index),
        "X_test": X_test
    }

    summary_rows.append({
        "Target": target_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Macro Precision": precision_score(y_test, y_pred, average="macro", zero_division=0),
        "Macro Recall": recall_score(y_test, y_pred, average="macro", zero_division=0),
        "Macro F1": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "Weighted F1": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    })

    print("\n" + "=" * 80)
    print("TARGET:", target_name)
    print("=" * 80)
    print(classification_report(y_test, y_pred, zero_division=0))

metrics_df = pd.DataFrame(summary_rows).set_index("Target")
print("\nSummary metrics:")
display(metrics_df.round(4))

## 6. Confusion matrices

A confusion matrix shows which labels the model confuses with one another. The specific-category model has many labels, so its chart may be large.

In [ ]:
for target_name, result in evaluation.items():
    labels = sorted(targets[target_name].unique())
    cm = confusion_matrix(result["y_test"], result["y_pred"], labels=labels)

    fig_size = max(8, min(18, len(labels) * 0.55))
    fig, ax = plt.subplots(figsize=(fig_size, fig_size * 0.8))
    sns.heatmap(
        cm,
        annot=(len(labels) <= 15),
        fmt="d",
        cmap="Reds",
        xticklabels=labels,
        yticklabels=labels,
        ax=ax
    )
    ax.set_title(f"Confusion Matrix — {target_name}")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 7. Inspect incorrect predictions

Review these examples to understand the model's weaknesses. Avoid publishing complaint text containing personal or confidential information.

In [ ]:
for target_name, result in evaluation.items():
    mistakes = pd.DataFrame({
        "complaint_text": result["X_test"],
        "actual": result["y_test"],
        "predicted": result["y_pred"]
    })
    mistakes = mistakes[mistakes["actual"] != mistakes["predicted"]]

    print(f"\n{target_name}: {len(mistakes)} incorrect predictions out of {len(result['y_test'])}")
    display(mistakes.head(10))

## 8. Predict a new mixed Bisaya/English complaint

Replace the sample below with a complaint you want to test. The outputs are model predictions based on the training data, not guaranteed or legally authoritative conclusions.

In [ ]:
def predict_complaint(complaint_text):
    complaint_text = str(complaint_text).strip()
    if not complaint_text:
        raise ValueError("Please enter complaint text.")

    result = {}
    for target_name, model in models.items():
        result[target_name] = model.predict([complaint_text])[0]
    return result

sample_complaint = (
    "Nipalit ko og item online but the seller took my payment and "
    "wala niya gi-deliver ang item bisan sige kog follow up."
)

prediction = predict_complaint(sample_complaint)
print("Complaint:", sample_complaint)
print("Predicted broad classification:", prediction["broad_classification"])
print("Predicted specific category:", prediction["specific_category"])

## 9. Save the trained models and label information

The ZIP contains both complete scikit-learn pipelines, which include the TF-IDF vectorizers and Linear SVM classifiers. These files can be loaded later by a Python API such as Flask or FastAPI.

**Important:** Keep the model files and this dataset secure. Only load trusted `.joblib` files because deserializing untrusted files can execute code.

In [ ]:
output_dir = "nlp_linear_svm_models"
os.makedirs(output_dir, exist_ok=True)

joblib.dump(models["broad_classification"], 
            os.path.join(output_dir, "broad_classification_svm.joblib"))
joblib.dump(models["specific_category"],
            os.path.join(output_dir, "specific_category_svm.joblib"))

metadata = {
    "algorithm": "TF-IDF word + character features with LinearSVC",
    "targets": {
        "broad_classification": sorted(data["classification"].unique().tolist()),
        "specific_category": sorted(data["category"].unique().tolist())
    },
    "text_column": "complaint_text",
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "training_rows": int(len(train_idx)),
    "testing_rows": int(len(test_idx)),
    "metrics": metrics_df.reset_index().to_dict(orient="records"),
    "note": "Predictions are decision support and require appropriate human review."
}

with open(os.path.join(output_dir, "metadata.json"), "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

zip_path = shutil.make_archive(output_dir, "zip", output_dir)
print("Saved model bundle:", zip_path)
files.download(zip_path)

## 10. Optional: export test-set predictions to Excel

This creates a file for model review, including the original complaint text, actual labels, and predicted labels. Store and share it only in accordance with your data privacy requirements.

In [ ]:
test_output = data.iloc[test_idx][
    [c for c in ["record_id", "complaint_text", "classification", "category", "language"] if c in data.columns]
].copy()

test_output["predicted_classification"] = evaluation["broad_classification"]["y_pred"].values
test_output["predicted_category"] = evaluation["specific_category"]["y_pred"].values

test_output["classification_correct"] = (
    test_output["classification"].astype(str) == test_output["predicted_classification"]
)
test_output["category_correct"] = (
    test_output["category"].astype(str) == test_output["predicted_category"]
)

review_file = "complaint_test_predictions.xlsx"
test_output.to_excel(review_file, index=False)
print("Created:", review_file)
files.download(review_file)

## Notes for interpreting results

- Report **accuracy, macro F1, and weighted F1**; do not rely on accuracy alone.
- The 32-category task is harder than the three-class broad classification task.
- Some categories may have relatively few examples. Collect more verified, representative examples for weak classes and retrain.
- Use a separate final test set or cross-validation when tuning hyperparameters. Do not tune repeatedly against the same test set and then report it as an unbiased final score.
- Linear SVM decision scores are **not calibrated probabilities**. Do not label them as confidence percentages without calibration.
- Since the dataset is mixed-language, manually review errors involving code-switching, spelling variation, and ambiguous complaint narratives.